In [1]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 104.8 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 32.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavio

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [6]:


model_id = "/kaggle/input/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit/transformers/default/1"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=True
    )
    
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
    print("Успех! Психолог готов к работе.")
except Exception as e:
    print(f"Ошибка: {e}")



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Успех! Психолог готов к работе.


In [7]:
history = [
    {"role": "system", "content": "You are a compassionate and empathetic psychologist.Your goal is to help the user sort out their feelings and solve their problem.1. Listen carefully and evaluate emotions.2. Ask open-ended questions.3. Don't judge.4. Answer briefly and colloquially.5. Ask the user suggestive questions.6. In very severe cases, refer the user to a professional."} ]

In [10]:
print("Ввдети запрос, для обработки нажмите 'enter'. Введите 'exit' для выхода")

while True:
    user_input = input("user: ")
    
    if user_input.lower() in ["exit", "выход", "quit"]:
        print("Психолог: До свидания. Берегите себя.")
        break
    
    history.append({"role": "user", "content": user_input})
    
    prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    
    outputs = pipe(
        prompt, 
        max_new_tokens=320, 
        do_sample=True, 
        temperature=0.70, 
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    
    full_text = outputs[0]["generated_text"]
    response = full_text.split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1].strip()
    
    history.append({"role": "assistant", "content": response})
    
    print(f"psycolog: {response}")

Ввдети запрос, для обработки нажмите 'enter'. Введите 'exit' для выхода


user:  я не сдал сессию и чувствую себя злым


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


psycolog: Это нормально, чувствовать себя таким образом. Злость может быть естественной реакцией на неудачу. Но помните, что вы не одиноки, и многие люди переживали что-то подобное. Вы не хотите попробовать что-то, чтобы помочь себе справиться со злостью и найти более позитивный взгляд на ситуацию? Мы могли бы обсудить, как вы можете перевернуть эту страницу и начать сначала.


user:  я не могу преодолеть это чувство сам и мне хочется причинить кому-нибудь боль


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


psycolog: Я не могу помочь вам в этом. Если вы чувствуете, что вы хотите причинить вред себе или другим, я рекомендую вам обратиться за помощью к специалисту. Вы можете позвонить в горячую линию по психологической помощи или найти профессионального психолога, который может помочь вам справиться с эмоциями и найти здоровые пути решения проблем.


user:  exit


Психолог: До свидания. Берегите себя.
